In [ ]:
REPO_URL = "https://github.com/30nikamvaishnavi/autonomous-driving-perception-stack.git"
BRANCH = "main"
DATA_DIR = None  # Override to use a different runtime dataset path.
OUT_DIR = None  # Override to change where outputs and checkpoints are saved.
WANDB_PROJECT = "adpstack"
MODEL_PATH = None  # Optional .pt path for export/benchmark notebooks.


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

SMOKE_STARTED_AT = time.monotonic()
IS_COLAB = (not Path("/kaggle").exists()) and (Path("/content").exists() or "COLAB_RELEASE_TAG" in os.environ)
IS_KAGGLE = Path("/kaggle").exists()
if IS_COLAB == IS_KAGGLE:
    raise RuntimeError("Unsupported or ambiguous runtime: use Google Colab or Kaggle.")

if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    runtime_root = Path("/content")
    data_default = Path("/content/drive/MyDrive/adpstack/data/bdd100k")
    output_default = Path("/content/drive/MyDrive/adpstack/outputs")
else:
    runtime_root = Path("/kaggle/working")
    data_default = Path("/kaggle/input")
    output_default = Path("/kaggle/working/outputs")

DATA_DIR = Path(DATA_DIR) if DATA_DIR is not None else data_default
OUT_DIR = Path(OUT_DIR) if OUT_DIR is not None else output_default
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

REPO_DIR = runtime_root / "autonomous-driving-perception-stack"
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
elif REPO_DIR.exists():
    raise FileExistsError(f"{REPO_DIR} exists but is not a Git checkout; remove/rename it and retry.")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,export]"],
    cwd=REPO_DIR,
    check=True,
)
import torch

print(f"Runtime: {'Colab' if IS_COLAB else 'Kaggle'}")
print(f"Repository: {REPO_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUT_DIR}")
print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached. Select a T4 GPU runtime in Colab or enable a GPU accelerator in Kaggle, then restart."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")


# 02 - YOLO fine-tuning on a Colab T4
Runtime > Change runtime type > T4 GPU.

In [ ]:
from pathlib import Path


def find_bdd_root(base: Path) -> Path:
    candidates = [base]
    if base.is_dir():
        candidates.extend(path for path in sorted(base.iterdir()) if path.is_dir())
    for candidate in candidates:
        if (candidate / "labels/det_20/det_train.json").is_file() and (candidate / "images/train").is_dir():
            return candidate
    raise FileNotFoundError(
        f"No BDD100K root under {base}; expected labels/det_20/det_train.json and images/train."
    )


BDD_ROOT = find_bdd_root(DATA_DIR)
YOLO_DATA_DIR = OUT_DIR / "bdd_yolo"
YOLO_DATASET = YOLO_DATA_DIR / "dataset.yaml"
print(f"BDD100K root: {BDD_ROOT}")

In [ ]:
from adpstack.data.bdd100k_to_yolo import convert_split, write_dataset_yaml

for split in ("train", "val"):
    converted = convert_split(
        BDD_ROOT / "labels" / "det_20" / f"det_{split}.json",
        BDD_ROOT / "images" / split,
        YOLO_DATA_DIR,
        split,
        subset=10000,
        copy_images=True,
    )
    if converted == 0:
        raise RuntimeError(f"No {split} images were converted from {BDD_ROOT}.")
write_dataset_yaml(YOLO_DATA_DIR)
print(f"YOLO dataset: {YOLO_DATASET}")

In [ ]:
import os

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
print(f"W&B project: {WANDB_PROJECT}. Login is optional; no credentials are stored in this notebook.")

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "adpstack.detection.train",
        "--set",
        f"data={YOLO_DATASET}",
        f"project={OUT_DIR / 'runs' / 'detect'}",
        "epochs=50", "model=yolo11n.pt", "name=yolo11n", "wandb.enabled=false",
    ],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "adpstack.detection.train",
        "--set",
        f"data={YOLO_DATASET}",
        f"project={OUT_DIR / 'runs' / 'detect'}",
        "epochs=50", "model=yolov8s.pt", "name=yolov8s", "wandb.enabled=false",
    ],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "adpstack.detection.evaluate",
        "--weights", str(OUT_DIR / "runs/detect/yolo11n/weights/best.pt"),
        "--data", str(YOLO_DATASET),
        "--labels-json", str(BDD_ROOT / "labels/det_20/det_val.json"),
        "--images-dir", str(YOLO_DATA_DIR / "images/val"),
        "--device", "0",
    ],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
BEST_PT = OUT_DIR / "runs/detect/yolo11n/weights/best.pt"
if not BEST_PT.is_file():
    raise FileNotFoundError(f"Training checkpoint not found: {BEST_PT}")
print(f"Best checkpoint saved at {BEST_PT}")